
# B 방식 (클린 버전): 제품 × 1명 프롬프트 생성
입력
- `/mnt/data/persona_attributes_weighted.jsonl`
- `/mnt/data/product_info_preprocessed.jsonl`

출력
- `/mnt/data/prompts_B.jsonl`
- `/mnt/data/prompts_B_preview.json`


In [ ]:

# =============================
# 0) CONFIG
# =============================
from pathlib import Path

PERSONA_JSONL = Path("/mnt/data/persona_attributes_weighted.jsonl")
PRODUCT_JSONL = Path("/mnt/data/product_info_preprocessed.jsonl")

OUT_JSONL     = Path("/mnt/data/prompts_B.jsonl")
OUT_PREVIEW   = Path("/mnt/data/prompts_B_preview.json")

# 제한 (None이면 전체)
LIMIT_PRODUCTS = None
LIMIT_PERSONAS = None

# 출력 옵션
ATTR_LIMIT = 60  # 페르소나 속성 최대 표기 개수

print("CONFIG loaded.")

In [ ]:

# =============================
# 1) Load data
# =============================
import json

personas = []
with open(PERSONA_JSONL, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            personas.append(json.loads(line))

products = []
with open(PRODUCT_JSONL, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            products.append(json.loads(line))

if LIMIT_PRODUCTS:
    products = products[:LIMIT_PRODUCTS]
if LIMIT_PERSONAS:
    personas = personas[:LIMIT_PERSONAS]

print("Loaded:", len(personas), "personas /", len(products), "products")

In [ ]:

# =============================
# 2) Helpers
# =============================
from typing import Dict, Any

def format_attributes_for_prompt(attrs: Dict[str, Any], limit:int=60) -> str:
    items = sorted(attrs.items(), key=lambda kv: kv[1].get("weight", 0.0), reverse=True)[:limit]
    lines = []
    for k, vw in items:
        v = vw.get("value", None)
        w = vw.get("weight", 0.0)
        v_str = "None" if v is None else str(v)
        lines.append(f"- {k}: {v_str} (w={w:.3f})")
    return "\n".join(lines)

def build_product_block(prod: Dict[str, Any]) -> str:
    if prod.get("prompt_block"):
        return prod["prompt_block"]
    lines = []
    if prod.get("product_id"): lines.append(f"- product_id: {prod['product_id']}")
    if prod.get("product_name"): lines.append(f"- 제품명: {prod['product_name']}")
    if prod.get("category"): lines.append(f"- 카테고리: {prod['category']}")
    if prod.get("features"): lines.append(f"- 주요 특징: {', '.join(prod['features'])}")
    if prod.get("targeted_consumer"): lines.append(f"- 타깃: {', '.join(prod['targeted_consumer'])}")
    if prod.get("release_info"): lines.append(f"- 출시일: {prod['release_info']}")
    if prod.get("price_text"): lines.append(f"- 기준 가격대: {prod['price_text']}")
    if prod.get("ad_model"): lines.append(f"- 광고모델: {prod['ad_model']}")
    if prod.get("advertise_info"): lines.append(f"- {prod['advertise_info']}")
    return "\n".join(lines)

def build_single_prompt(product: Dict[str, Any], persona: Dict[str, Any], attr_limit:int=60) -> str:
    product_block = build_product_block(product)
    meta = persona.get("meta", {}) or {}
    cluster = meta.get("cluster", "")
    label = meta.get("label", "")
    desc = meta.get("desc", meta.get("Description",""))

    # precompute attribute and cluster strings (avoid backslashes inside f-exprs)
    attrs_str = format_attributes_for_prompt(persona.get("attributes", {}), limit=attr_limit)
    if (cluster or label or desc):
        cluster_block = "- cluster: " + str(cluster) + "\n" + "- label: " + str(label) + "\n" + "- desc: " + str(desc)
    else:
        cluster_block = "- cluster: N/A"

    # assemble with plain concatenation (no embedded \ in {} exprs)
    parts = []
    parts.append("[역할]")
    parts.append("당신은 한국 소비자 데이터 분석가이자 마케팅 전문가입니다.")
    parts.append("아래의 \"제품 정보\"와 \"페르소나\"를 바탕으로,")
    parts.append("이 페르소나가 해당 제품의 구매자로서 성립하는 **싱글 턴** 페르소나 JSON을 생성하세요.")
    parts.append("")
    parts.append("[제품 정보]")
    parts.append(product_block)
    parts.append("")
    parts.append("[페르소나]")
    parts.append(f"- id: {persona.get('persona_key','')}")
    parts.append("- 속성(가중치 합=1):")
    parts.append(attrs_str)
    parts.append("")
    parts.append("[클러스터 컨텍스트]")
    parts.append(cluster_block)
    parts.append("")
    parts.append("[규칙]")
    parts.append("- '클러스터 컨텍스트'는 배경 지침으로만 사용합니다. 속성 가중치(합=1)와 충돌 시 '속성 가중치'를 우선합니다.")
    parts.append("- 2024-07 ~ 2025-06 월별로 구매확률(prob 0~1)과 예상수량(qty 정수)을 제시합니다.")
    parts.append("- 추석/설, 광고/프로모션/계절성을 반영합니다.")
    parts.append("- **반드시 아래 JSON 스키마를 출력**하고, 불필요한 설명 문장은 출력하지 마세요.")
    parts.append("")
    parts.append("[출력 스키마(JSON)]")
    parts.append("{")
    parts.append(f'  "persona_id": "p_{{product_id_or_name}}_{persona.get("persona_key","")}",')
    parts.append(f'  "product_name": "{product.get("product_name","")}",')
    parts.append(f'  "product_id": "{product.get("product_id","")}",')
    parts.append(f'  "segment_ref": "{persona.get("persona_key","")}",')
    parts.append('  "attributes": { "{{속성명}}": {"value": "<값>", "weight": <0~1> }, "...": "..." },')
    parts.append('  "purchase_pattern": {')
    parts.append('    "avg_purchase_prob": <0~1>,')
    parts.append('    "avg_purchase_qty": <int>,')
    parts.append('    "seasonality": {"추석": "+x%", "설": "+y%"},')
    parts.append('    "promotion_effect": "광고/프로모션 노출 시 +z%"')
    parts.append('  },')
    parts.append('  "forecast_12mo": {')
    parts.append('    "2024-07": {"prob": <0~1>, "qty": <int>},')
    parts.append('    "...": {},')
    parts.append('    "2025-06": {"prob": <0~1>, "qty": <int>}')
    parts.append('  }')
    parts.append("}")
    return "\n".join(parts)

In [ ]:

# =============================
# 3) Build & save
# =============================
import json
from pathlib import Path

records = []
for prod in products:
    pid_or_name = prod.get("product_id") or (prod.get("product_name","") or "").replace(" ", "_")
    for persona in personas:
        prompt_text = build_single_prompt({**prod, "product_id_or_name": pid_or_name}, persona, attr_limit=ATTR_LIMIT)
        records.append({
            "product": {"product_id_or_name": pid_or_name, "product_name": prod.get("product_name")},
            "persona": {"persona_key": persona.get("persona_key")},
            "prompt": prompt_text
        })

with open(OUT_JSONL, "w", encoding="utf-8") as f:
    for r in records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

Path(OUT_PREVIEW).write_text(json.dumps(records[:3], ensure_ascii=False, indent=2), encoding="utf-8")
print("Saved:", OUT_JSONL, "bytes=", OUT_JSONL.stat().st_size)
len(records)